# Credit Card Approval Prediction

## Overview
This notebook demonstrates an end-to-end machine learning pipeline for predicting credit card approvals using a subset of the UCI Credit Card Approval dataset. The pipeline includes:

1. Data loading and exploration
2. Data cleaning and imputation
3. Feature encoding and scaling
4. Training baseline models (Logistic Regression, Decision Tree)
5. Training ensemble models (Random Forest, Gradient Boosting)
6. Model evaluation (ROC-AUC, Precision-Recall curves)
7. Fairness and bias considerations

The code is structured using reusable modules in the `src/` directory.

In [ ]:
# Import required libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Import custom modules
sys.path.append('..')
from src.data_loader import load_data
from src.preprocessing import DataPreprocessor
from src.model_training import ModelTrainer, split_data
from src.evaluation import ModelEvaluator

# Set display options
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

## 1. Data Loading and Exploration

We'll load the credit card approval dataset and explore its basic characteristics.

In [ ]:
# Load the data
df = load_data('../data/cc_approvals.data')

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Check data types
print("Data types:")
print(df.dtypes)
print("\nDataset info:")
df.info()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_percent
})
print("Missing values per column:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Check target variable distribution
print("Target variable distribution:")
print(df['A16'].value_counts())
print("\nTarget proportions:")
print(df['A16'].value_counts(normalize=True))

In [ ]:
# Visualize missing data
plt.figure(figsize=(12, 6))
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
plt.bar(range(len(missing_data)), missing_data.values)
plt.xticks(range(len(missing_data)), missing_data.index, rotation=45)
plt.ylabel('Number of Missing Values')
plt.title('Missing Values by Feature')
plt.tight_layout()
plt.show()

## 2. Data Cleaning and Imputation

We'll handle missing values using appropriate imputation strategies:
- Numeric features: impute with mean
- Categorical features: impute with most frequent value

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Identify column types
numeric_cols, categorical_cols = preprocessor.identify_column_types(df)
print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"\nCategorical columns ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Clean and impute missing values
df_clean = preprocessor.clean_and_impute(df, fit=True)
print("Missing values after imputation:")
print(df_clean.isnull().sum().sum())

## 3. Feature Encoding and Scaling

We'll encode categorical variables using Label Encoding and scale all features using StandardScaler.

In [ ]:
# Preprocess the data (encoding and scaling)
X_scaled, y = preprocessor.preprocess(df, fit=True)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nTarget classes: {np.unique(y)}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.3)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

## 4. Baseline Models

We'll train two baseline models:
1. Logistic Regression
2. Decision Tree

In [ ]:
# Initialize model trainer
trainer = ModelTrainer(random_state=42)

# Train Logistic Regression
print("Training Logistic Regression...")
logreg = trainer.train_logistic_regression(X_train, y_train)
logreg_train_score = logreg.score(X_train, y_train)
logreg_test_score = logreg.score(X_test, y_test)
print(f"  Training accuracy: {logreg_train_score:.4f}")
print(f"  Test accuracy: {logreg_test_score:.4f}")

In [ ]:
# Train Decision Tree
print("Training Decision Tree...")
tree = trainer.train_decision_tree(X_train, y_train)
tree_train_score = tree.score(X_train, y_train)
tree_test_score = tree.score(X_test, y_test)
print(f"  Training accuracy: {tree_train_score:.4f}")
print(f"  Test accuracy: {tree_test_score:.4f}")

## 5. Ensemble Models

We'll train ensemble models to improve performance:
1. Random Forest
2. Gradient Boosting

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf = trainer.train_random_forest(X_train, y_train)
rf_train_score = rf.score(X_train, y_train)
rf_test_score = rf.score(X_test, y_test)
print(f"  Training accuracy: {rf_train_score:.4f}")
print(f"  Test accuracy: {rf_test_score:.4f}")

In [ ]:
# Train Gradient Boosting
print("Training Gradient Boosting...")
gb = trainer.train_gradient_boosting(X_train, y_train)
gb_train_score = gb.score(X_train, y_train)
gb_test_score = gb.score(X_test, y_test)
print(f"  Training accuracy: {gb_train_score:.4f}")
print(f"  Test accuracy: {gb_test_score:.4f}")

## 6. Model Evaluation

We'll evaluate all models using comprehensive metrics including:
- Accuracy, Precision, Recall, F1-score
- ROC-AUC scores
- Precision-Recall curves
- Confusion matrices

In [ ]:
# Initialize evaluator and evaluate all models
evaluator = ModelEvaluator()
all_models = trainer.get_all_models()
results = evaluator.evaluate_all_models(all_models, X_test, y_test)

In [ ]:
# Print detailed metrics
evaluator.print_metrics()

In [ ]:
# Compare all models
comparison_df = evaluator.compare_models()
print("\nModel Comparison:")
comparison_df

In [ ]:
# Plot ROC curves
evaluator.plot_roc_curves(y_test)
plt.show()

In [ ]:
# Plot Precision-Recall curves
evaluator.plot_precision_recall_curves(y_test)
plt.show()

In [ ]:
# Plot confusion matrices
evaluator.plot_confusion_matrices(figsize=(16, 4))
plt.show()

## 7. Model Insights and Feature Importance

Let's examine which features are most important for the best performing models.

In [ ]:
# Get feature importance from Random Forest
feature_names = [f'A{i}' for i in range(1, 16)]
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances - Random Forest")
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.xlabel("Features")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
for i in range(5):
    print(f"{i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}")

## 8. Fairness and Bias Considerations

### Important Ethical Considerations

When deploying credit card approval models in production, it's crucial to consider:

#### 1. Protected Attributes
The dataset has been anonymized, but in real-world scenarios, features might correlate with protected attributes like:
- Race/Ethnicity
- Gender
- Age
- Marital status

#### 2. Disparate Impact
- **Definition**: When a facially neutral policy has a disproportionately adverse effect on protected groups
- **Concern**: Even without explicit protected attributes, proxy features can perpetuate bias
- **Action**: Regularly audit model predictions across different demographic groups

#### 3. Historical Bias
- **Issue**: Models trained on historical data may learn and amplify past discriminatory practices
- **Example**: If historical approval rates were biased, the model may perpetuate these biases
- **Mitigation**: 
  - Use fairness-aware algorithms
  - Consider reweighting training samples
  - Apply post-processing fairness corrections

#### 4. Fairness Metrics to Monitor
- **Demographic Parity**: Equal approval rates across groups
- **Equal Opportunity**: Equal true positive rates across groups
- **Predictive Parity**: Equal precision across groups
- **Calibration**: Predicted probabilities match actual outcomes across groups

#### 5. Transparency and Explainability
- Applicants have a right to understand why they were denied
- Regulatory requirements (e.g., FCRA in the US, GDPR in EU) mandate explainability
- Use interpretable models or explanation techniques (SHAP, LIME)

#### 6. Regular Monitoring
- Continuously monitor model performance across subgroups
- Track changes in approval rates over time
- Implement automated alerts for unexpected disparities

#### 7. Human Oversight
- Models should assist, not replace, human decision-making
- Establish appeal processes for denied applications
- Regular review by ethics boards or fairness committees

### Recommended Actions
1. Conduct regular fairness audits using tools like:
   - Fairlearn (Microsoft)
   - AI Fairness 360 (IBM)
   - What-If Tool (Google)

2. Document model limitations and known biases

3. Establish clear governance and accountability structures

4. Provide training to stakeholders on responsible AI use

5. Stay updated with evolving regulations and best practices

## Summary

### Key Findings:
1. **Best Model**: Based on ROC-AUC and balanced performance metrics
2. **Data Quality**: Successfully handled missing values through imputation
3. **Model Performance**: All models showed good performance, with ensemble methods typically outperforming baseline models

### Recommendations:
1. Deploy the best performing model with continuous monitoring
2. Implement fairness audits before production deployment
3. Set up A/B testing framework for model validation
4. Establish model retraining schedule to maintain performance
5. Create explainability dashboards for stakeholders

### Next Steps:
- Hyperparameter tuning using GridSearchCV or RandomizedSearchCV
- Try additional ensemble methods (XGBoost, LightGBM)
- Implement SMOTE for handling class imbalance if needed
- Add fairness constraints to the optimization objective
- Deploy model with monitoring and logging infrastructure